Cell 0 — load the table we already saved


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "data" / "processed" / "merged_with_features.csv")

df["loan_purpose"] = df["loan_purpose"].replace({"Personaal": "Personal"})
df["default"] = df["default"].astype(int)

print(df.shape)
print(df["default"].value_counts())

(50000, 36)
default
0    45703
1     4297
Name: count, dtype: int64


Cell 1 — drop leakage / IDs / extra location cols

In [2]:
drop_cols = [
    "loan_id", "cust_id",
    "principal_outstanding", "disbursal_date", "installment_start_dt",
    "net_disbursement", "gst", "processing_fee",
    "delinquent_months", "total_loan_months", "total_dpd",
    "sanction_amount",
    "city", "state", "zipcode",
]

model_df = df.drop(columns=drop_cols)
print(model_df.columns.tolist())
print(model_df.shape)
print(model_df.dtypes)

['loan_purpose', 'loan_type', 'loan_amount', 'loan_tenure_months', 'bank_balance_at_application', 'default', 'age', 'gender', 'marital_status', 'employment_status', 'income', 'number_of_dependants', 'residence_type', 'years_at_current_address', 'number_of_open_accounts', 'number_of_closed_accounts', 'enquiry_count', 'credit_utilization_ratio', 'loan_to_income', 'delinquency_ratio', 'avg_dpd_per_delinquency']
(50000, 21)
loan_purpose                       str
loan_type                          str
loan_amount                      int64
loan_tenure_months               int64
bank_balance_at_application      int64
default                          int64
age                              int64
gender                             str
marital_status                     str
employment_status                  str
income                           int64
number_of_dependants             int64
residence_type                     str
years_at_current_address         int64
number_of_open_accounts       

Cell 2 — X and y

In [3]:
y = model_df["default"]
X = model_df.drop(columns=["default"])

print("y rate:", y.mean())
print("X shape:", X.shape)

y rate: 0.08594
X shape: (50000, 20)


Cell 3 — dummy-encode categories (simple v1)

In [4]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
print("categorical:", cat_cols)

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print("X after dummies:", X.shape)
print(X.columns.tolist())

categorical: ['loan_purpose', 'loan_type', 'gender', 'marital_status', 'employment_status', 'residence_type']
X after dummies: (50000, 23)
['loan_amount', 'loan_tenure_months', 'bank_balance_at_application', 'age', 'income', 'number_of_dependants', 'years_at_current_address', 'number_of_open_accounts', 'number_of_closed_accounts', 'enquiry_count', 'credit_utilization_ratio', 'loan_to_income', 'delinquency_ratio', 'avg_dpd_per_delinquency', 'loan_purpose_Education', 'loan_purpose_Home', 'loan_purpose_Personal', 'loan_type_Unsecured', 'gender_M', 'marital_status_Single', 'employment_status_Self-Employed', 'residence_type_Owned', 'residence_type_Rented']


Cell 4 — train / test split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print("train default rate:", y_train.mean())
print("test default rate :", y_test.mean())

(40000, 23) (10000, 23)
train default rate: 0.08595
test default rate : 0.0859


Cell 5 — scale numbers (needed for logistic)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [8]:
print("NaN per column:")
print(X.isna().sum()[X.isna().sum() > 0])

print("\ninf per column:")
print(np.isinf(X.select_dtypes(include="number")).sum())

NaN per column:
loan_to_income    8
dtype: int64

inf per column:
loan_amount                    0
loan_tenure_months             0
bank_balance_at_application    0
age                            0
income                         0
number_of_dependants           0
years_at_current_address       0
number_of_open_accounts        0
number_of_closed_accounts      0
enquiry_count                  0
credit_utilization_ratio       0
loan_to_income                 0
delinquency_ratio              0
avg_dpd_per_delinquency        0
dtype: int64


In [9]:
print("total_loan_months == 0:", (df["total_loan_months"] == 0).sum())
print("income == 0:", (df["income"] == 0).sum())
print("delinquency_ratio NaN:", df["delinquency_ratio"].isna().sum())
print("loan_to_income NaN:", df["loan_to_income"].isna().sum())
print("avg_dpd NaN:", df["avg_dpd_per_delinquency"].isna().sum())

total_loan_months == 0: 0
income == 0: 8
delinquency_ratio NaN: 0
loan_to_income NaN: 8
avg_dpd NaN: 0


In [10]:
df["delinquency_ratio"] = df["delinquent_months"] / df["total_loan_months"].replace(0, pd.NA)
df["delinquency_ratio"] = df["delinquency_ratio"].fillna(0)

df["loan_to_income"] = df["loan_amount"] / df["income"].replace(0, pd.NA)
df["loan_to_income"] = df["loan_to_income"].fillna(0)

df["avg_dpd_per_delinquency"] = df["total_dpd"] / df["delinquent_months"].replace(0, pd.NA)
df["avg_dpd_per_delinquency"] = df["avg_dpd_per_delinquency"].fillna(0)

df[["loan_to_income", "delinquency_ratio", "avg_dpd_per_delinquency"]] = (
    df[["loan_to_income", "delinquency_ratio", "avg_dpd_per_delinquency"]]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(df[["loan_to_income", "delinquency_ratio", "avg_dpd_per_delinquency"]].isna().sum())

loan_to_income             0
delinquency_ratio          0
avg_dpd_per_delinquency    0
dtype: int64


In [11]:
print("NaN in X_train_sc:", np.isnan(X_train_sc).sum())
print("NaN in X_test_sc :", np.isnan(X_test_sc).sum())

NaN in X_train_sc: 8
NaN in X_test_sc : 0


In [12]:
print(X_train.isna().sum()[X_train.isna().sum() > 0])
print("rows with any NaN:", X_train.isna().any(axis=1).sum())
display(X_train[X_train.isna().any(axis=1)])

loan_to_income    8
dtype: int64
rows with any NaN: 8


,loan_amount,loan_tenure_months,bank_balance_at_application,age,income,number_of_dependants,years_at_current_address,number_of_open_accounts,number_of_closed_accounts,enquiry_count,...,avg_dpd_per_delinquency,loan_purpose_Education,loan_purpose_Home,loan_purpose_Personal,loan_type_Unsecured,gender_M,marital_status_Single,employment_status_Self-Employed,residence_type_Owned,residence_type_Rented
4950,0,32,0,42,0,3,13,3,0,7,...,0.000000,False,True,False,False,True,False,False,False,False
46640,0,28,0,28,0,4,24,2,0,6,...,0.000000,False,False,False,False,False,False,False,False,True
13356,0,24,0,34,0,1,2,2,2,3,...,5.681818,True,False,False,False,False,True,True,True,False
47428,0,50,0,27,0,0,19,4,0,4,...,5.800000,True,False,False,False,True,True,True,True,False
2278,0,13,0,46,0,2,26,4,2,7,...,0.000000,False,False,False,False,True,False,True,False,True
45224,0,17,0,43,0,4,30,3,1,7,...,5.300000,False,False,True,True,True,False,False,False,False
11092,0,26,0,39,0,0,17,2,1,5,...,0.000000,False,True,False,False,False,True,False,False,True
27457,0,44,0,48,0,3,5,4,1,7,...,0.000000,False,True,False,False,False,False,True,True,False


In [13]:
nan_pos = np.argwhere(np.isnan(X_train_sc))
print(nan_pos[:20])
print("columns:", [X_train.columns[j] for j in np.unique(nan_pos[:, 1])])

[[ 1236    11]
 [ 3290    11]
 [ 4212    11]
 [10832    11]
 [20837    11]
 [23106    11]
 [29809    11]
 [36164    11]]
columns: ['loan_to_income']


In [14]:
print("NaN before fill:", X.isna().sum().sum())
X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)
print("NaN after fill:", X.isna().sum().sum())

NaN before fill: 8
NaN after fill: 0


In [15]:
print(np.isnan(X_train_sc).sum(), np.isnan(X_test_sc).sum())

8 0


In [16]:
# 1) kill remaining NaN / inf on the dummy-encoded X
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
print("NaN in X:", X.isna().sum().sum())   # must print 0

# 2) split AGAIN from this clean X
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) scale AGAIN
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(np.isnan(X_train_sc).sum(), np.isnan(X_test_sc).sum())  # must print 0 0

# 4) fit
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_sc, y_train)
print("intercept:", log_reg.intercept_[0])

NaN in X: 0
0 0
intercept: -6.21682677548296


In [17]:
print(X_train.isna().sum()[X_train.isna().sum() > 0])
print(X_train.columns[X_train.isna().any()].tolist())

Series([], dtype: int64)
[]


In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # overwrite
X_test_sc = scaler.transform(X_test)         # overwrite

print(np.isnan(X_train_sc).sum(), np.isnan(X_test_sc).sum())

0 0


In [19]:
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_sc, y_train)
print("fitted, intercept:", log_reg.intercept_[0])

fitted, intercept: -6.21682677548296


Cell 6 — fit logistic regression

In [21]:
# Coefficients#


In [22]:
coef = pd.Series(log_reg.coef_[0], index=X_train.columns).sort_values()

print("=== risky (positive, default UP) ===")
print(coef.tail(10))

=== risky (positive, default UP) ===
loan_tenure_months          0.081397
loan_purpose_Education      0.396898
loan_purpose_Personal       0.445596
loan_type_Unsecured         0.445596
enquiry_count               0.601400
avg_dpd_per_delinquency     0.615770
residence_type_Rented       0.731385
delinquency_ratio           2.247207
loan_to_income              3.650959
credit_utilization_ratio    4.474518
dtype: float64


In [23]:
# Metrics

In [24]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [25]:
y_prob = log_reg.predict_proba(X_test_sc)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

In [26]:
print("AUC:", roc_auc_score(y_test, y_prob))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

AUC: 0.9839350371536653
Confusion matrix:
 [[8447  694]
 [  45  814]]
              precision    recall  f1-score   support

           0      0.995     0.924     0.958      9141
           1      0.540     0.948     0.688       859

    accuracy                          0.926     10000
   macro avg      0.767     0.936     0.823     10000
weighted avg      0.956     0.926     0.935     10000



Save for the app later

In [27]:
import joblib
from pathlib import Path

ART = Path("..").resolve() / "artifacts"
ART.mkdir(exist_ok=True)

joblib.dump(
    {
        "model": log_reg,
        "scaler": scaler,
        "features": X_train.columns.tolist(),
    },
    ART / "model_data.joblib",
)
print("saved", ART / "model_data.joblib")

saved D:\Downloads\namaste_ankit_sql\ml_project_2\artifacts\model_data.joblib


$\text{score} = 300 + (1 - p) \times 600$


$p=1$ → 300 (worst). $p=0$ → 900 (best).

In [28]:
#credit score

In [29]:
def pd_to_score_rating(p):
    score = int(round(300 + (1 - p) * 600))
    score = min(900, max(300, score))
    if score <= 499:
        rating = "Poor"
    elif score <= 649:
        rating = "Average"
    elif score <= 749:
        rating = "Good"
    else:
        rating = "Excellent"
    return score, rating

# sanity
for p in [0.02, 0.10, 0.30, 0.60, 0.90]:
    print(p, pd_to_score_rating(p))

# a few test rows
preview = pd.DataFrame({
    "y_true": y_test.values,
    "pd": y_prob,
})
preview["score"], preview["rating"] = zip(*preview["pd"].map(pd_to_score_rating))
display(preview.head(10))
print(preview.groupby("rating")["y_true"].mean())

0.02 (888, 'Excellent')
0.1 (840, 'Excellent')
0.3 (720, 'Good')
0.6 (540, 'Average')
0.9 (360, 'Poor')


,y_true,pd,score,rating
0,0,3.132485e-01,712,Good
1,0,9.751117e-01,315,Poor
2,0,4.854736e-05,900,Excellent
3,0,6.139497e-04,900,Excellent
4,0,2.046757e-07,900,Excellent
5,0,7.996107e-07,900,Excellent
6,0,9.410048e-01,335,Poor
7,0,4.193221e-08,900,Excellent
8,0,3.070120e-05,900,Excellent
9,0,1.782190e-02,889,Excellent


rating
Average      0.113757
Excellent    0.001368
Good         0.061728
Poor         0.623016
Name: y_true, dtype: float64


In [30]:
import joblib
from pathlib import Path

ART = Path("..").resolve() / "artifacts"
joblib.dump(
    {
        "model": log_reg,
        "scaler": scaler,
        "features": X_train.columns.tolist(),
    },
    ART / "model_data.joblib",
)
print("saved", ART / "model_data.joblib")

saved D:\Downloads\namaste_ankit_sql\ml_project_2\artifacts\model_data.joblib


SyntaxError: invalid syntax (2166113234.py, line 1)

Slim feature list

In [32]:
slim_num = [
    "age", "income", "loan_amount", "loan_tenure_months",
    "loan_to_income", "avg_dpd_per_delinquency", "delinquency_ratio",
    "credit_utilization_ratio", "number_of_open_accounts",
]
slim_cat = ["residence_type", "loan_purpose", "loan_type"]

slim = df[slim_num + slim_cat + ["default"]].copy()
slim = slim.replace([np.inf, -np.inf], np.nan).fillna(0)



In [33]:
y_s = slim["default"].astype(int)
X_s = pd.get_dummies(slim.drop(columns=["default"]), columns=slim_cat, drop_first=True)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42, stratify=y_s
)

scaler_s = StandardScaler()
Xs_train_sc = scaler_s.fit_transform(Xs_train)
Xs_test_sc = scaler_s.transform(Xs_test)

log_s = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_s.fit(Xs_train_sc, ys_train)

ps = log_s.predict_proba(Xs_test_sc)[:, 1]
print("slim AUC:", roc_auc_score(ys_test, ps))
print(confusion_matrix(ys_test, (ps >= 0.5).astype(int)))
print("slim columns:", X_s.columns.tolist())

slim AUC: 0.9830867311104176
[[8426  715]
 [  46  813]]
slim columns: ['age', 'income', 'loan_amount', 'loan_tenure_months', 'loan_to_income', 'avg_dpd_per_delinquency', 'delinquency_ratio', 'credit_utilization_ratio', 'number_of_open_accounts', 'residence_type_Owned', 'residence_type_Rented', 'loan_purpose_Education', 'loan_purpose_Home', 'loan_purpose_Personal', 'loan_type_Unsecured']


In [34]:
import joblib
from pathlib import Path

ART = Path("..").resolve() / "artifacts"
joblib.dump(
    {
        "model": log_s,
        "scaler": scaler_s,
        "features": X_s.columns.tolist(),
    },
    ART / "model_data.joblib",
)

['D:\\Downloads\\namaste_ankit_sql\\ml_project_2\\artifacts\\model_data.joblib']